# 🩺 Medical RAG: WHO Hypertension Guideline

This notebook presents a unified, evidence-grounded Retrieval Augmented Generation (RAG) pipeline for the WHO guideline on the pharmacological treatment of hypertension in adults.

### ✨ Key Features:
*   **OCR-aware text cleaning**: Robust processing removes malformed HTML, LaTeX remnants, and boilerplate while preserving critical medical content.
*   **Section-aware chunking**: Ensures focused and bounded text passages for precise retrieval.
*   **Hybrid retrieval**: Combines normalized FAISS similarity search with transparent lexical reranking and maximal marginal relevance for improved medical-term matching and evidence diversity.
*   **LLM-powered Answer Generation**: Utilizes a local `Qwen` model for synthesizing structured, emoji-rich, and evidence-grounded answers.
*   **Direct LLM Formatting**: Responses are directly formatted by the Qwen model with confidence scores and traceability.
*   **No API Key Required**: Entire LLM inference runs locally, ensuring privacy and eliminating external costs.

**Tech Stack:** `Python`, `pandas`, `faiss`, `sentence-transformers`, `transformers`, `torch`, `Hugging Face`

**Dataset:** WHO Guideline on the pharmacological treatment of hypertension in adults



## 🗒️ Table of Contents

1.  [**1. Introduction**](#1-introduction)
2.  [**2. Setup and Configuration**](#2-setup-and-configuration)
    *   [2.1. Install Dependencies](#2-1-install-dependencies)
    *   [2.2. Imports](#2-2-imports)
    *   [2.3. Configuration](#2-3-configuration)
3.  [**3. Data Processing**](#3-data-processing)
    *   [3.1. Load OCR JSON](#3-1-load-ocr-json)
    *   [3.2. Text Cleaning](#3-2-text-cleaning)
    *   [3.3. Section Detection](#3-3-section-detection)
    *   [3.4. Structural Noise Removal](#3-4-structural-noise-removal)
    *   [3.5. Flatten Blocks](#3-5-flatten-blocks)
    *   [3.6. Section-Aware Chunking](#3-6-section-aware-chunking)
    *   [3.7. Metadata Generation](#3-7-metadata-generation)
    *   [3.8. Dataset Validation](#3-8-dataset-validation)
    *   [3.9. Save JSON Dataset](#3-9-save-json-dataset)
4.  [**4. Retrieval Augmented Generation (RAG)**](#4-retrieval-augmented-generation-rag)
    *   [4.1. Embedding Model](#4-1-embedding-model)
    *   [4.2. Embedding Generation](#4-2-embedding-generation)
    *   [4.3. Vector Database (FAISS)](#4-3-vector-database-faiss)
    *   [4.4. Retrieval](#4-4-retrieval)
    *   [4.5. Retrieval Evaluation](#4-5-retrieval-evaluation)
    *   [4.6. Grounded, Evidence-Based Answer Generation](#4-6-grounded-evidence-based-answer-generation)
    *   [4.7. Local Medical LLM for Answer Synthesis](#4-7-local-medical-llm-for-answer-synthesis)
    *   [4.8. Safety / Refusal Layer](#4-8-safety---refusal-layer)
    *   [4.9. End-to-End RAG Function](#4-9-end-to-end-rag-function)
5.  [**5. Demonstration and Evaluation**](#5-demonstration-and-evaluation)
    *   [5.1. Demo Questions](#5-1-demo-questions)
    *   [5.2. Evaluation Summary](#5-2-evaluation-summary)
6.  [**6. Conclusion & Next Steps**](#6-conclusion--next-steps)

# 1. Introduction

This section introduces the project and its goals, providing an overview of the medical RAG pipeline for the WHO hypertension guideline.

# 2. Setup and Configuration

This section covers the necessary setup steps, including dependency installation, library imports, and project configuration.

### Dependency Installation

This cell ensures that all necessary Python packages are installed, including `faiss-cpu`, `sentence-transformers`, `transformers`, `accelerate`, and `torch`. It uses a helper function to only install missing packages, making the notebook robust across different environments.

## 2.1. Install Dependencies
We'll install only the necessary, lightweight, and Kaggle-friendly packages here.

In [63]:
import importlib.util
import subprocess
import sys


def ensure_packages(packages, required=True):
    """Install only packages that are not already available."""
    missing = [
        package
        for package, module_name in packages.items()
        if importlib.util.find_spec(module_name) is None
    ]
    if missing:
        print("Installing missing packages:", ", ".join(missing))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as error:
            if required:
                raise
            print(f"Optional package install skipped: {error}")


ensure_packages({"faiss-cpu": "faiss"}, required=True)
ensure_packages({"sentence-transformers": "sentence_transformers"}, required=False)
# Add core LLM dependencies here for consolidation
ensure_packages({"transformers": "transformers", "accelerate": "accelerate", "torch": "torch"}, required=True)
print("Dependency check complete.")

Dependency check complete.


## 2.2. Imports
All necessary libraries are imported here for a streamlined workflow.

In [64]:
import os
import re
import json
import warnings
from collections import Counter

import numpy as np

warnings.filterwarnings("ignore") # Suppress unnecessary warnings for cleaner output

### Core Imports

All essential libraries required for data processing, natural language understanding, and machine learning are imported here. This includes `os`, `re`, `json`, `warnings`, and `collections` for general utilities, and `numpy` for numerical operations.

## 2.3. Configuration
This section centralizes all constants and file paths, serving as the single source of truth for our project settings.

### Project Configuration and Path Resolution

This cell defines critical project constants, file paths, and configuration parameters. The `resolve_uploaded_file` helper function ensures compatibility with both Kaggle and Colab environments by checking multiple possible file locations. Key settings for cleaning, chunking, and retrieval are centralized here.

In [65]:
import os


def resolve_uploaded_file(*filenames):
    """Support Kaggle/Colab uploads and the local attached_assets folder."""
    candidates = []
    for filename in filenames:
        candidates.extend(
            [
                f"/content/{filename}",
                f"/kaggle/working/{filename}",
                os.path.join("attached_assets", filename),
                filename,
            ]
        )
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        f"Could not find any of {filenames}. Checked: {', '.join(candidates)}"
    )


# --- Project Configuration ---
JSON_PATH = resolve_uploaded_file(
    "DOC-20260818-WA0020.pdf-1.6.json",
    "DOC-20260818-WA0020.pdf-1.6_1787058053818.json",
)
MD_PATH = resolve_uploaded_file(
    "DOC-20260818-WA0020.pdf-VL-1.6 - Copy.md", # Corrected filename
    "DOC-20260818-WA0020.pdf-VL-1.6.md",
    "DOC-20260818-WA0020.pdf-VL-1.6_1787058053820.md",
)

OUTPUT_DIR = os.path.join(os.getcwd(), "output")
CHUNKS_JSONL_PATH = os.path.join(OUTPUT_DIR, "hypertension_chunks_metadata.json")
FAISS_INDEX_PATH = os.path.join(OUTPUT_DIR, "hypertension_faiss.index")

DOCUMENT_NAME = "Guideline for the pharmacological treatment of hypertension in adults"
SOURCE_URL = "https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content"

# Cleaning parameters
REPEATED_HEADER = DOCUMENT_NAME
CONTENTS_PAGE = 5
IGNORED_BLOCK_LABELS = {
    "header",
    "footer",
    "header_image",
    "footer_image",
    "footnote",
    "aside_text",
}
SECTION_LABELS = {"paragraph_title", "section_header", "doc_title"}
NON_CONTENT_PAGES = {4, 5, 6, 61}

# Smaller, section-aware chunks reduce topic mixing and improve evidence precision.
TARGET_WORDS = 360
MAX_WORDS = 450
OVERLAP_WORDS = 60

PRIMARY_EMBEDDING_MODEL = "sentence-transformers/embeddinggemma-300m-medical"
FALLBACK_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 16

# Hybrid retrieval: dense semantics + transparent lexical evidence matching.
DEFAULT_TOP_K = 5
RETRIEVAL_CANDIDATES = 24
DENSE_WEIGHT = 0.65
LEXICAL_WEIGHT = 0.35
MMR_LAMBDA = 0.78

# Confidence is reported as a calibrated retrieval signal, not a medical diagnosis.
MIN_CONFIDENCE_FOR_CAUTION = 52
MIN_CONFIDENCE_FOR_ALLOWED = 68

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print("Input JSON:", JSON_PATH)

Configuration loaded.
Input JSON: /content/DOC-20260818-WA0020.pdf-1.6.json


# 3. Data Processing

This section details the entire data processing pipeline, from loading the OCR output to generating validated and chunked datasets for retrieval. It covers cleaning, section detection, noise removal, chunking, metadata generation, and validation.

## 3.1. Load OCR JSON
This step uses the already-processed PaddleOCR-VL JSON output as our primary data source.

In [66]:
import os

def load_ocr_json(path):
    # Load the JSON file from PaddleOCR-VL, including error handling
    if not os.path.exists(path):
        raise FileNotFoundError(f"OCR JSON file not found at: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list) or len(data) == 0:
        raise ValueError("Unexpected OCR JSON structure: expected a non-empty list of pages.")
    return data

raw_pages = load_ocr_json(JSON_PATH)
print("Loaded pages:", len(raw_pages))

Loaded pages: 61


## 3.2. Text Cleaning
Our text cleaning process combines robust methods, including HTML stripping and boilerplate removal, drawing from various patterns identified in both submissions.

### Text Cleaning Functions and Execution

This block defines a suite of text cleaning functions designed to remove boilerplate, HTML, LaTeX remnants, and other noise from the OCR-extracted text. It also includes heuristics to identify and filter out non-meaningful text blocks, ensuring that only high-quality content proceeds to the next stages. The `extract_cleaned_pages` function applies these cleaning steps to the raw OCR pages.

In [67]:
import html
import re
import unicodedata
from collections import Counter

# --- Regex constants ---
_BOILERPLATE = [
    r"(?im)^\s*NICE National Institute for Health and Care Excellence\s*$",
    r"(?im)^\s*(?:Published|Last updated):\s*\d+\s*[A-Za-z]+\s*\d{4}\s*$",
    r"(?im)^\s*\N{COPYRIGHT SIGN}\s*NICE\s*\d{4}.*$",
    r"(?im)^\s*\N{COPYRIGHT SIGN}\s*World Health Organization\s*\d{4}\s*$",
    r"(?im)^\s*#{0,6}\s*\N{COPYRIGHT SIGN}\s*World Health Organization\s*\d{4}\s*$",
    r"(?im)^\s*Guideline for the pharmacological treatment of hypertension in adults\s*$",
    r"(?im)^\s*WHO Guidelines Approved by the Guidelines Review Committee\s*$",
    r"(?im)^\s*ISBN\s+\d{3}-\d-\d{3}-\d{5}-\d\s*$",
    r"(?im)^\s*978-\d-\d{3}-\d{5}-\d\s*$",
    r"(?im)^\s*\d+\s*of\s*\d+\s*$",
    r"(?im)^\s*\d{1,2}\s*\|\s*WHO\s*$",
]

_PICO_MARKERS = [
    "population", "intervention", "comparator", "comparison", "outcome",
    "pico", "grade", "certainty of evidence", "evidence to decision",
    "risk of bias", "annex", "summary of findings",
]

_LATEX_MAP = {
    r"\\geq?": "\N{GREATER-THAN OR EQUAL TO}",
    r"\\leq?": "\N{LESS-THAN OR EQUAL TO}",
    r"\\pm": "\N{PLUS-MINUS SIGN}",
    r"\\times": "\N{MULTIPLICATION SIGN}",
    r"\\alpha": "\N{GREEK SMALL LETTER ALPHA}",
    r"\\beta": "\N{GREEK SMALL LETTER BETA}",
    r"\\gamma": "\N{GREEK SMALL LETTER GAMMA}",
}


def is_pico_annex_block(text: str) -> bool:
    """True if text looks like a PICO/annex methodology table."""
    if not text:
        return False
    lowered = text.lower()
    hits = sum(m in lowered for m in _PICO_MARKERS)
    pipe_density = lowered.count("|") / max(len(lowered), 1)
    return (hits >= 3 and len(lowered) < 1200) or (pipe_density > 0.01 and hits >= 2)


def strip_latex(text: str) -> str:
    for pat, repl in _LATEX_MAP.items():
        text = re.sub(pat, repl, text, flags=re.I)
    text = re.sub(r"\\(?:text|mathrm|mathbf)\\s*\\{([^\\{}]*)\\}", r"\1", text)
    text = re.sub(r"\\underline\\s*\\{([^\\{}]*)\\}", r"\1", text)
    return re.sub(r"\\[a-zA-Z]+", " ", text.replace("$", ""))


def strip_html(text: str) -> str:
    for pat in [
        r"</?\s*(?:br|tr|td|th|table|tbody|thead|div|span|p)\b[^>]*/?>",
        r"(?:style|class)\s*=\s*['\"][^'\"]*['\"]",
        r"[a-zA-Z-]+\s*:\s*[a-zA-Z0-9%#\- ]+;\s*'?'?\"?",
        r"break-word;?'?",
    ]:
        text = re.sub(pat, " ", text, flags=re.I)
    return re.sub(r"<[^>]*>", " ", text)


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "").replace("\ufffd", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = strip_html(text)
    text = strip_latex(text)
    text = text.replace("\u2013", "-").replace("\u2014", "-").replace("\u2212", "-")
    text = text.replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'")
    for pat in _BOILERPLATE:
        text = re.sub(pat, "", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    text = re.sub(r"\s*([<>\u2264\u2265=])\s*", r" \1 ", text)
    text = re.sub(r"\s+", " ", text.replace("\n", " ")).strip()
    return "" if is_pico_annex_block(text) else text


def is_meaningful(text: str) -> bool:
    alnum = re.findall(r"[A-Za-z0-9]", text)
    if not text or len(alnum) < 3:
        return False
    if len(text) > 12 and len(alnum) / len(text) < 0.18:
        return False
    return not re.fullmatch(r"(.)\1{3,}", text)


def _sig(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


def extract_cleaned_pages(pages):
    candidates, sigs = [], Counter()
    for page_number, page in enumerate(pages, start=1):
        page_cands = []
        for block in page.get("prunedResult", {}).get("parsing_res_list", []):
            label = block.get("block_label", "")
            if label in IGNORED_BLOCK_LABELS:
                continue
            content = clean_text(block.get("block_content", ""))
            if not is_meaningful(content):
                continue
            sig = _sig(content)
            if sig:
                sigs[sig] += 1
            page_cands.append({"label": label, "text": content,
                               "block_order": block.get("block_order"), "signature": sig})
        candidates.append((page_number, page_cands))

    header_sig = _sig(REPEATED_HEADER)
    cleaned, prev = [], None
    for page_number, page_cands in candidates:
        blocks = []
        for block in page_cands:
            sig = block.pop("signature")
            if (sigs[sig] >= 4 and len(sig.split()) <= 18) or sig == header_sig or sig == prev:
                continue
            prev = sig
            blocks.append(block)
        cleaned.append({"page_number": page_number, "blocks": blocks})
    return cleaned


cleaned_pages = extract_cleaned_pages(raw_pages)
print("Cleaned pages:", len(cleaned_pages))

Cleaned pages: 61


## 3.3. Section Detection
This step dynamically assigns a `section_title` to each block based on identified heading blocks within the document structure.

In [68]:
def detect_sections(cleaned_pages):
    # Identifies and assigns the current section title to each block
    sectioned_pages = []
    current_section = "Introduction" # Default section

    for page in cleaned_pages:
        new_blocks = []
        for block in page["blocks"]:
            label = block["label"]
            text = block["text"]

            # If the block is a section label, update the current section title
            if label in SECTION_LABELS:
                section_title = re.sub(r"^#+\s*", "", text).strip()
                if section_title:
                    current_section = section_title

            new_blocks.append({
                "page_number": page["page_number"],
                "section_title": current_section,
                "label": label,
                "text": text,
                "block_order": block.get("block_order"),
            })
        sectioned_pages.append({"page_number": page["page_number"], "blocks": new_blocks})
    return sectioned_pages

sectioned_pages = detect_sections(cleaned_pages)
print("Pages with section labels attached:", len(sectioned_pages))

Pages with section labels attached: 61


## 3.4. Structural Noise Removal

**Merge Decision:** The original notebook included two versions of this step: an aggressive one that removed blocks repeated on 5+ pages, and a safer one that only removed the known Contents page and repeated header string. The aggressive version was unused and risked removing legitimate medical content. Therefore, we've kept only the **safe version** to ensure content integrity.

In [69]:
def remove_structural_noise(sectioned_pages):
    safe_pages = []
    for page in sectioned_pages:
        if page["page_number"] in NON_CONTENT_PAGES:
            continue
        new_blocks = []
        for block in page["blocks"]:
            text = block["text"].strip()
            if text == REPEATED_HEADER:
                continue
            new_blocks.append({**block, "text": text})
        if new_blocks:
            safe_pages.append({"page_number": page["page_number"], "blocks": new_blocks})
    return safe_pages


safe_cleaned_pages = remove_structural_noise(sectioned_pages)
print("Original pages:", len(cleaned_pages))
print("Pages after safe cleaning:", len(safe_cleaned_pages))

Original pages: 61
Pages after safe cleaning: 52


## 3.5. Flatten Blocks
This step flattens the structured pages into a single, ordered list of blocks, preparing them for chunking.

In [70]:
def flatten_blocks(safe_pages):
    # Converts the list of pages with blocks into a single flat list of all blocks
    all_blocks = []
    for page in safe_pages:
        for block in page["blocks"]:
            all_blocks.append({
                "page_number": block["page_number"],
                "section_title": block["section_title"],
                "label": block["label"],
                "text": block["text"],
                "block_order": block.get("block_order"),
            })
    return all_blocks

all_blocks = flatten_blocks(safe_cleaned_pages)
print("Total blocks:", len(all_blocks))

Total blocks: 494


## 3.6. Section-Aware Chunking

**Merge Decision:** We're retaining the section-aware, word-based chunke that splits on section boundaries and tracks page numbers. The alternative `RecursiveCharacterTextSplitter` (character-based, no section awareness) was dropped because it loses critical section-level traceability, which is a key project requirement.

In [71]:
def count_words(text):
    return len(re.findall(r"\b\w[\w'-]*\b", text, flags=re.UNICODE))


def split_long_block(text, limit=MAX_WORDS):
    if count_words(text) <= limit:
        return [text]
    parts, buf, buf_w = [], [], 0
    for sent in re.split(r"(?<=[.!?])\s+", text):
        w = count_words(sent)
        if w > limit:
            if buf:
                parts.append(" ".join(buf).strip())
                buf, buf_w = [], 0
            words = sent.split()
            for i in range(0, len(words), limit):
                parts.append(" ".join(words[i:i + limit]))
        else:
            if buf and buf_w + w > limit:
                parts.append(" ".join(buf).strip())
                buf, buf_w = [], 0
            buf.append(sent)
            buf_w += w
    if buf:
        parts.append(" ".join(buf).strip())
    return [p for p in parts if p]


def create_chunks(blocks):
    chunks, buf, words, section = [], [], 0, None

    def flush():
        nonlocal buf, words
        if not buf:
            return
        text = "\n\n".join(t for t, _ in buf).strip()
        chunks.append({
            "text": text,
            "section_title": section,
            "page_numbers": sorted({p for _, p in buf}),
            "word_count": count_words(text),
        })
        # overlap tail
        tail, need = [], OVERLAP_WORDS
        for t, p in reversed(buf):
            w = t.split()
            if w:
                take = min(len(w), need)
                tail.insert(0, (" ".join(w[-take:]), p))
                need -= take
                if need <= 0:
                    break
        buf = tail
        words = sum(count_words(t) for t, _ in buf)

    for block in blocks:
        sec, page = block["section_title"], block["page_number"]
        if section is not None and sec != section:
            flush()
            buf, words = [], 0
        section = sec
        for bt in split_long_block(block["text"].strip()):
            bw = count_words(bt)
            if not bw:
                continue
            if buf and words + bw > MAX_WORDS:
                flush()
            if words + bw > MAX_WORDS:
                buf, words = [], 0
            buf.append((bt, page))
            words += bw
            if words >= TARGET_WORDS:
                flush()

    if buf and words > OVERLAP_WORDS:
        flush()
    print("Total chunks:", len(chunks))
    return chunks


raw_chunks = create_chunks(all_blocks)

Total chunks: 113


## 3.7. Metadata Generation
We use a canonical schema to ensure consistent metadata across the entire pipeline, adhering to our data standardization requirements.

In [72]:
def add_metadata(chunks):
    seen = {}
    for ch in chunks:
        text = clean_text(ch["text"])
        if not is_meaningful(text):
            continue
        key = " ".join(text.split()).lower()
        if key in seen:
            seen[key]["page_numbers"] = sorted(
                set(seen[key]["page_numbers"]) | set(ch["page_numbers"])
            )
            seen[key]["text"] = text
        else:
            seen[key] = {
                **ch,
                "text": text,
                "page_numbers": list(ch["page_numbers"]),
                "word_count": count_words(text),
            }

    return [
        {
            "chunk_id": f"chunk_{i:03d}",
            "document_name": DOCUMENT_NAME,
            "text": c["text"],
            "section_title": c["section_title"],
            "page_numbers": c["page_numbers"],
            "word_count": c["word_count"],
            "source_url": SOURCE_URL,
        }
        for i, c in enumerate(seen.values(), start=1)
    ]


final_chunks = add_metadata(raw_chunks)
print("Total chunks with metadata:", len(final_chunks))
if final_chunks:
    print(json.dumps(final_chunks[1], indent=2, ensure_ascii=False))

Total chunks with metadata: 111
{
  "chunk_id": "chunk_002",
  "document_name": "Guideline for the pharmacological treatment of hypertension in adults",
  "text": "WHO Guidelines Review Committee Secretariat, and Nathan Ford, Chair of the Guidelines Review Committee are gratefully acknowledged for their technical support throughout the process. Thanks are also due to Alma Alic from the Department of Compliance, Risk Management and Ethics for her support in the assessment of declarations of interests. Sheila Nakpil from the Department of NCDs provided logistical support. WHO would like to recognize the voices of persons with lived experiences with hypertension whom we heard from through consultation during development of this guideline.",
  "section_title": "Acknowledgements",
  "page_numbers": [
    7
  ],
  "word_count": 85,
  "source_url": "https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content"
}


## 3.8. Dataset Validation
This step ensures that each chunk meets our structural and content requirements, verifying data integrity before further processing.

In [73]:
REQUIRED_FIELDS = [
    "chunk_id", "document_name", "text", "section_title",
    "page_numbers", "word_count", "source_url",
]


def validate_chunks(chunks):
    errors, seen_ids, seen_text = [], set(), set()
    for i, ch in enumerate(chunks, start=1):
        errors += [f"Chunk {i}: missing field '{f}'" for f in REQUIRED_FIELDS if f not in ch]
        text = ch.get("text", "").strip()
        if not is_meaningful(text):
            errors.append(f"Chunk {i}: empty or OCR-noise text")
        if ch.get("word_count", 0) > MAX_WORDS:
            errors.append(f"Chunk {i}: word_count {ch['word_count']} exceeds MAX_WORDS")
        cid = ch.get("chunk_id")
        if cid in seen_ids:
            errors.append(f"Chunk {i}: duplicate chunk_id")
        if text in seen_text:
            errors.append(f"Chunk {i}: duplicate chunk text")
        seen_ids.add(cid)
        seen_text.add(text)
    return errors


validation_errors = validate_chunks(final_chunks)
if validation_errors:
    print(f"Validation FAILED - {len(validation_errors)} issue(s):")
    for err in validation_errors[:20]:
        print(" -", err)
    raise ValueError("Chunk validation failed; stop before embedding.")
print(f"Validation passed - all {len(final_chunks)} chunks are well-formed.")

Validation passed - all 111 chunks are well-formed.


## 3.9. Save JSON Dataset
We're saving our processed chunks into a JSON file, a highly reusable format for downstream applications.

In [74]:
# Saves the final, validated chunks into a JSON file
with open(CHUNKS_JSONL_PATH, "w", encoding="utf-8") as f:
    json.dump(final_chunks, f, indent=2, ensure_ascii=False)

print("Dataset saved to:", CHUNKS_JSONL_PATH)
print("Total records:", len(final_chunks))

Dataset saved to: /content/output/hypertension_chunks_metadata.json
Total records: 111


# 4. Retrieval Augmented Generation (RAG)

This section outlines the core RAG components, including embedding generation, vector database construction, retrieval mechanisms, and grounded answer generation.

## 4.1. Embedding Model

**Why this model:** `embeddinggemma-300m-medical` is a `sentence-transformers` model specifically fine-tuned for retrieving medical text. It's well-suited for our hypertension guideline task and efficient enough to run on Kaggle's CPU.

Since it's a gated model, it might fail to load in certain environments. To maintain pipeline robustness, we've implemented an **automatic fallback** to `all-MiniLM-L6-v2`, a smaller, ungated general-purpose model, ensuring the notebook remains runnable end-to-end regardless of the environment.

In [75]:
import hashlib

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None


class HashingEmbeddingModel:
    """Dependency-free emergency fallback."""

    def __init__(self, dimension=384):
        self.dimension = dimension

    def get_sentence_embedding_dimension(self):
        return self.dimension

    def encode(self, texts, batch_size=EMBEDDING_BATCH_SIZE,
               normalize_embeddings=True, show_progress_bar=False, **kwargs):
        vectors = []
        for text in texts:
            v = np.zeros(self.dimension, dtype="float32")
            for token in re.findall(r"[a-z0-9]+", str(text).lower()):
                d = hashlib.md5(token.encode()).digest()
                idx = int.from_bytes(d[:4], "little") % self.dimension
                v[idx] += 1.0 if d[4] % 2 else -1.0
            if normalize_embeddings:
                n = np.linalg.norm(v)
                if n:
                    v /= n
            vectors.append(v)
        return np.asarray(vectors, dtype="float32")


def load_embedding_model():
    if SentenceTransformer is None:
        print("sentence-transformers unavailable; using hashing fallback.")
        return HashingEmbeddingModel(), "hashing-fallback-384"
    for name, kwargs in [(PRIMARY_EMBEDDING_MODEL, {"trust_remote_code": True}),
                         (FALLBACK_EMBEDDING_MODEL, {})]:
        try:
            model = SentenceTransformer(name, **kwargs)
            print(f"Loaded embedding model: {name}")
            return model, name
        except Exception as e:
            print(f"Failed to load {name}: {e}")
    print("All embedding models failed; using hashing fallback.")
    return HashingEmbeddingModel(), "hashing-fallback-384"


embedding_model, embedding_model_name = load_embedding_model()
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print("Embedding dimension:", embedding_dim)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/embeddinggemma-300m-medical
Embedding dimension: 768


## 4.2. Embedding Generation
This step converts each text chunk into a numerical vector (embedding), crucial for semantic search capabilities.

In [76]:
def encode_texts(model, texts, is_query=False):
    # Encodes texts into embeddings, optionally using specific prompts for query/document distinction
    prompt_kwarg = {}
    # embeddinggemma models support prompt_name="query"/"document" for better retrieval quality
    try:
        if is_query:
            prompt_kwarg = {"prompt_name": "query"}
        else:
            prompt_kwarg = {"prompt_name": "document"}
        return model.encode(
            texts, batch_size=EMBEDDING_BATCH_SIZE,
            normalize_embeddings=True, show_progress_bar=False, **prompt_kwarg
        )
    except TypeError:
        # Fallback model does not support prompt_name, so encode plainly
        return model.encode(
            texts, batch_size=EMBEDDING_BATCH_SIZE,
            normalize_embeddings=True, show_progress_bar=False
        )

if not final_chunks:
    raise ValueError("No chunks available to embed — check the chunking stage.")

chunk_texts = [c["text"] for c in final_chunks]
chunk_embeddings = np.asarray(encode_texts(embedding_model, chunk_texts, is_query=False), dtype="float32")
print("Embeddings shape:", chunk_embeddings.shape)

Embeddings shape: (111, 768)


## 4.3. Vector Database (FAISS)

**Design Decision:** We've chosen FAISS for its lightweight nature and efficiency, making it ideal for a Kaggle environment. It avoids the overhead of server-based solutions like Chroma, aligning with our goal of a simple, self-contained notebook.

In [77]:
import faiss


def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index


faiss_index = build_faiss_index(chunk_embeddings)
faiss.write_index(faiss_index, FAISS_INDEX_PATH)
chunk_lookup = {i: final_chunks[i] for i in range(len(final_chunks))}

print("FAISS index built with", faiss_index.ntotal, "vectors.")
print("Index saved to:", FAISS_INDEX_PATH)

FAISS index built with 111 vectors.
Index saved to: /content/output/hypertension_faiss.index


## 4.4. Retrieval
This function queries the FAISS index to fetch the most semantically similar document chunks for a given input query.

In [78]:
import faiss

STOPWORDS = {
    "a", "an", "and", "are", "be", "for", "from", "in", "is", "of", "on",
    "or", "that", "the", "to", "what", "when", "with", "which", "who",
}
RECOMMENDATION_HINTS = (
    "recommendation", "treatment", "management", "target",
    "threshold", "initiation", "follow-up", "monitoring",
)
ANCHOR_PHRASES = (
    "first line", "blood pressure", "target blood pressure",
    "pharmacological treatment", "reassessed",
)
PICO_ANNEX_PENALTY = 0.35


def lexical_tokens(text):
    return {t for t in re.findall(r"[a-z0-9]+", text.lower()) if len(t) > 1 and t not in STOPWORDS}


chunk_token_sets = {i: lexical_tokens(c["text"]) for i, c in chunk_lookup.items()}
document_frequency = Counter(t for tokens in chunk_token_sets.values() for t in tokens)
total_chunks = max(len(chunk_token_sets), 1)
idf = {t: np.log((total_chunks + 1) / (f + 1)) + 1 for t, f in document_frequency.items()}
chunk_is_pico = {
    i: is_pico_annex_block(c["text"]) or "annex" in c["section_title"].lower()
    for i, c in chunk_lookup.items()
}


def lexical_relevance(query, idx):
    q_tokens = lexical_tokens(query)
    if not q_tokens:
        return 0.0

    chunk = chunk_lookup[idx]
    text_lower, section_lower = chunk["text"].lower(), chunk["section_title"].lower()
    q_lower, q_norm = query.lower().strip(), query.lower().strip().replace("-", " ")

    weighted = lambda tokens: sum(idf.get(t, 1.0) for t in tokens)
    score = weighted(q_tokens & chunk_token_sets[idx]) / max(weighted(q_tokens), 1e-9)
    score += 0.22 * len(q_tokens & lexical_tokens(section_lower)) / len(q_tokens)

    if len(q_norm) > 12 and q_norm in text_lower.replace("-", " "):
        score += 0.20
    if any(p in q_norm and p in section_lower for p in ANCHOR_PHRASES):
        score += 0.12
    if len(q_lower) > 12 and q_lower in text_lower:
        score += 0.20
    if any(h in section_lower for h in RECOMMENDATION_HINTS):
        score += 0.08

    return min(1.0, score)


def retrieve(query: str, top_k: int = DEFAULT_TOP_K):
    """Retrieves the most relevant document chunks based on a query."""
    if not query or not query.strip():
        raise ValueError("Query must not be empty.")

    top_k = max(1, min(int(top_k), len(final_chunks)))
    candidate_k = min(len(final_chunks), max(RETRIEVAL_CANDIDATES, top_k * 4))

    query_embedding = np.asarray(encode_texts(embedding_model, [query], is_query=True), dtype="float32")
    dense_scores, indices = faiss_index.search(query_embedding, candidate_k)
    dense_by_idx = {int(i): float(s) for s, i in zip(dense_scores[0], indices[0]) if i != -1}

    lexical_scores = {idx: lexical_relevance(query, idx) for idx in chunk_lookup}
    top_lexical = sorted(lexical_scores, key=lexical_scores.get, reverse=True)[:candidate_k]
    candidate_indices = set(dense_by_idx) | set(top_lexical)

    candidates = []
    for idx in candidate_indices:
        dense = dense_by_idx.get(idx, float(np.dot(query_embedding[0], chunk_embeddings[idx])))
        dense_norm = float(np.clip((dense + 1.0) / 2.0, 0.0, 1.0))
        combined = DENSE_WEIGHT * dense_norm + LEXICAL_WEIGHT * lexical_scores[idx]
        if chunk_is_pico.get(idx):
            combined *= PICO_ANNEX_PENALTY
        candidates.append({
            "index": idx, "score": combined, "dense_score": dense,
            "lexical_score": lexical_scores[idx], "is_pico_annex": chunk_is_pico.get(idx, False),
        })

    # MMR: pick candidates that balance relevance with novelty vs. already-picked ones.
    selected, remaining = [], candidates
    while remaining and len(selected) < top_k:
        def mmr(c):
            redundancy = max(
                (float(np.dot(chunk_embeddings[c["index"]], chunk_embeddings[s["index"]])) for s in selected),
                default=0.0,
            )
            return MMR_LAMBDA * c["score"] - (1 - MMR_LAMBDA) * redundancy
        best = max(remaining, key=mmr)
        selected.append(best)
        remaining.remove(best)

    selected.sort(key=lambda c: c["score"], reverse=True)
    return [
        {**{k: c[k] for k in ("score", "dense_score", "lexical_score", "is_pico_annex")},
         **{k: chunk_lookup[c["index"]][k] for k in ("chunk_id", "text", "section_title", "page_numbers", "source_url")}}
        for c in selected
    ]


sample_results = retrieve("What are the pharmacological treatments for hypertension?", top_k=3)
for r in sample_results:
    print(
        f"[combined={r['score']:.3f} dense={r['dense_score']:.3f} lexical={r['lexical_score']:.3f} "
        f"pico={r['is_pico_annex']}] {r['chunk_id']} | {r['section_title']} | pages {r['page_numbers']}"
    )

[combined=0.852 dense=0.546 lexical=1.000 pico=False] chunk_107 | 2 Is any laboratory testing necessary prior to initiation or during titration of pharmacological treatments | pages [56, 57]
[combined=0.808 dense=0.596 lexical=0.825 pico=False] chunk_111 | 10 In adults with hypertension given pharmacological treatment, when should blood pressure be reassessed? | pages [59, 60]
[combined=0.796 dense=0.559 lexical=0.825 pico=False] chunk_109 | 2 Is any laboratory testing necessary prior to initiation or during titration of pharmacological treatments | pages [58, 59]


## 4.5. Retrieval Evaluation

Since we lack ground-truth relevance judgments for this dataset, this evaluation is **heuristic** and based on keyword overlap. It's important to note that this is not a substitute for formal Precision@K metrics with labeled relevant chunk IDs.

In [79]:
# This remains a heuristic retrieval check because no human relevance labels
# were provided. It now measures both result-level precision and token coverage.
RETRIEVAL_TEST_SET = [
    {
        "query": "What is the recommended first-line pharmacological treatment for hypertension?",
        "expected_keywords": ["first-line", "pharmacological", "hypertension", "treatment"],
    },
    {
        "query": "When should blood pressure be reassessed after initiating pharmacological treatment?",
        "expected_keywords": ["blood pressure", "reassessed", "pharmacological", "treatment"],
    },
    {
        "query": "What are the adverse effects of antihypertensive medications?",
        "expected_keywords": ["adverse effects", "antihypertensive", "medications"],
    },
    {
        "query": "What is the target blood pressure for adults with hypertension?",
        "expected_keywords": ["target", "blood pressure", "hypertension"],
    },
]


def keyword_coverage(text, expected_keywords):
    text_lower = text.lower()
    hits = sum(1 for keyword in expected_keywords if keyword.lower() in text_lower)
    return hits / max(len(expected_keywords), 1)


def heuristic_precision_at_k(test_set, top_k=DEFAULT_TOP_K):
    per_query_results = []
    for case in test_set:
        results = retrieve(case["query"], top_k=top_k)
        coverages = [keyword_coverage(result["text"], case["expected_keywords"]) for result in results]
        relevant = [c for c in coverages if c >= 0.25]
        coverage = np.mean(coverages) if results else 0.0
        per_query_results.append(
            {
                "query": case["query"],
                "heuristic_precision_at_k": round(len(relevant) / max(len(results), 1), 2),
                "mean_keyword_coverage": round(float(coverage), 2),
                "top_result_section": results[0]["section_title"] if results else None,
            }
        )
    return per_query_results


eval_results = heuristic_precision_at_k(RETRIEVAL_TEST_SET, top_k=DEFAULT_TOP_K)
mean_precision = float(np.mean([r["heuristic_precision_at_k"] for r in eval_results]))
mean_coverage = float(np.mean([r["mean_keyword_coverage"] for r in eval_results]))

print("NOTE: These are heuristic metrics, not human-labeled Precision@K.")
for result in eval_results:
    print(f"- {result['query']}")
    print(
        f"    Precision@{DEFAULT_TOP_K}: {result['heuristic_precision_at_k']:.2f} | "
        f"keyword coverage: {result['mean_keyword_coverage']:.2f} | "
        f"top section: {result['top_result_section']}"
    )
print(f"\nMean heuristic Precision@{DEFAULT_TOP_K}: {mean_precision:.2f}")
print(f"Mean keyword coverage: {mean_coverage:.2f}")

NOTE: These are heuristic metrics, not human-labeled Precision@K.
- What is the recommended first-line pharmacological treatment for hypertension?
    Precision@5: 1.00 | keyword coverage: 0.90 | top section: 4. RECOMMENDATION ON DRUG CLASSES TO BE USED AS FIRST-LINE AGENTS
- When should blood pressure be reassessed after initiating pharmacological treatment?
    Precision@5: 1.00 | keyword coverage: 0.80 | top section: 10 In adults with hypertension given pharmacological treatment, when should blood pressure be reassessed?
- What are the adverse effects of antihypertensive medications?
    Precision@5: 1.00 | keyword coverage: 0.67 | top section: Strong recommendation, high-certainty evidence
- What is the target blood pressure for adults with hypertension?
    Precision@5: 1.00 | keyword coverage: 0.87 | top section: 6. RECOMMENDATIONS ON TARGET BLOOD PRESSURE

Mean heuristic Precision@5: 1.00
Mean keyword coverage: 0.81


## 4.5.2 Human Retrieval Evaluation

In [80]:
# Human-labeled ground truth based on WHO guideline
HUMAN_EVAL_SET = [
    {
        "query": "What is the recommended first-line pharmacological treatment for hypertension?",
        "expected_ids": ["chunk_009", "chunk_038"],
    },
    {
        "query": "What is the target blood pressure for adults with hypertension?",
        "expected_ids": ["chunk_011", "chunk_047"],
    },
    {
        "query": "At what blood pressure threshold should pharmacological treatment be initiated?",
        "expected_chunk_ids": ["chunk_006", "chunk_028"],
        "expected_ids": ["chunk_006", "chunk_028"],
    },
    {
        "query": "Can pharmacists or nurses provide pharmacological treatment for hypertension?",
        "expected_ids": ["chunk_013", "chunk_053"],
    },
]

def human_precision_at_k(test_set, top_k=DEFAULT_TOP_K):
    precisions = []
    hits = []

    for case in test_set:
        results = retrieve(case["query"], top_k=top_k)
        retrieved_ids = [r["chunk_id"] for r in results]

        # Check matching chunks
        correct = [cid for cid in retrieved_ids if cid in case["expected_ids"]]

        p_at_k = len(correct) / len(results) if results else 0.0
        hit = 1.0 if len(correct) > 0 else 0.0

        precisions.append(p_at_k)
        hits.append(hit)

        print(f"- {case['query']}")
        print(f"    Precision@{top_k}: {p_at_k:.2f} | Hit: {hit:.0f} | Top ID: {retrieved_ids[0] if retrieved_ids else None}")

    print(f"\nMean Human Precision@{top_k}: {np.mean(precisions):.2f}")
    print(f"Mean Hit Rate@{top_k}: {np.mean(hits):.2f}")

human_precision_at_k(HUMAN_EVAL_SET, top_k=DEFAULT_TOP_K)

- What is the recommended first-line pharmacological treatment for hypertension?
    Precision@5: 0.20 | Hit: 1 | Top ID: chunk_009
- What is the target blood pressure for adults with hypertension?
    Precision@5: 0.20 | Hit: 1 | Top ID: chunk_011
- At what blood pressure threshold should pharmacological treatment be initiated?
    Precision@5: 0.40 | Hit: 1 | Top ID: chunk_028
- Can pharmacists or nurses provide pharmacological treatment for hypertension?
    Precision@5: 0.40 | Hit: 1 | Top ID: chunk_013

Mean Human Precision@5: 0.30
Mean Hit Rate@5: 1.00


## 4.6. Grounded, Evidence-Based Answer Generation

**Update:** The extractive / string-slicing fallback has been removed. Context synthesis now runs a dedup + PICO/annex-filter pass (`build_evidence_block`, `build_clean_context`) before anything reaches the LLM, and generated prose is cleaned of truncation and citation artifacts (`clean_generated_text`) before it is shown or formatted. The local medical LLM in Section 18b is the sole answer generation engine -- see Section 20 for how a missing/failed LLM is surfaced explicitly instead of degrading to raw chunk text.

## 4.7. Local Medical LLM for Answer Synthesis
We load a small, open-source, **medically fine-tuned** LLM locally via Hugging Face `transformers`.

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` is used for answer synthesis, loaded with `torch.float16` precision for efficient inference on a free Colab T4 GPU. No API key, no external cost.

In [62]:
LLM_MAX_NEW_TOKENS = 350
LLM_TEMPERATURE = 0.1

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2-1.5B-Instruct"

_llm_model = None
_llm_tokenizer = None
_llm_model_name_loaded = None

try:
    _llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    _llm_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    _llm_model_name_loaded = MODEL_ID
    print(f"{MODEL_ID} model and tokenizer loaded successfully within losewnLGjaiA.")
except Exception as e:
    _llm_model_name_loaded = f"{MODEL_ID} model failed to load: {e}"
    print(f"Failed to load {MODEL_ID} model in losewnLGjaiA: {e}")

def medical_llm_generate_fn(prompt: str) -> str:
    """llm_generate_fn implementation backed by the local quantized model."""
    if _llm_model is None or _llm_tokenizer is None:
        raise RuntimeError("Local LLM is not loaded.")
    inputs = _llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(_llm_model.device)
    with torch.no_grad():
        output_ids = _llm_model.generate(
            **inputs,
            max_new_tokens=LLM_MAX_NEW_TOKENS,
            temperature=LLM_TEMPERATURE,
            do_sample=LLM_TEMPERATURE > 0,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4,
            pad_token_id=_llm_tokenizer.pad_token_id or _llm_tokenizer.eos_token_id,
            eos_token_id=_llm_tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_text = _llm_tokenizer.decode(generated_ids, skip_special_tokens=True)
    return generated_text.strip()

LLM_AVAILABLE = _llm_model is not None and _llm_tokenizer is not None
print("Local medical LLM available:", LLM_AVAILABLE, "| model:", _llm_model_name_loaded)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2-1.5B-Instruct model and tokenizer loaded successfully within losewnLGjaiA.
Local medical LLM available: True | model: Qwen/Qwen2-1.5B-Instruct


### Generate Clinical Response

This function orchestrates the final answer generation process. It takes the user's query, retrieved evidence chunks, and confidence scores, then uses the local LLM (`_llm_model`) to synthesize a grounded, evidence-based response in a structured format. It also handles post-processing of the LLM's output and formats traceability information.

In [81]:
import re

SYSTEM_PROMPT = """You are an expert clinical AI assistant synthesizing medical evidence into clear, actionable responses.

STRICT RULES:
1. Synthesize ORIGINAL, complete sentences from context. Do NOT copy raw chunks directly.
2. NEVER output tags like [chunk_001] in your text.
3. NEVER write non-answers like "See evidence below".

REQUIRED FORMAT:
🎯 **Direct Clinical Answer**:
[1–2 bold, complete sentences directly answering the query]

💊 **Key Guidelines & Recommendations**:
- **[Inline Bold Title]**: [Synthesized clinical step]

⚠️ **Considerations & Risk Factors**:
- [Relevant comorbidity, contraindication, or demographic rule]"""

def clean_chunk_text(text: str) -> str:
    text = re.sub(r"\[chunk_\w+(?:,\s*p\.\s*\d+)?\]", "", text)
    text = re.sub(r"\b[PICO]\s*\|", "", text)
    return re.sub(r"\s+", " ", text).strip()

def generate_clinical_response(query: str, retrieved_chunks: list[dict], confidence: dict) -> str:
    cleaned_context_list, traceability_lines = [], []
    for idx, chunk in enumerate(retrieved_chunks, 1):
        cleaned_context_list.append(f"Source [{idx}]: {clean_chunk_text(chunk.get('text', ''))}")
        chunk_id = chunk.get("chunk_id", f"chunk_{idx:03d}")
        page = ", ".join(map(str, chunk.get("page_numbers", ["N/A"])))
        section = chunk.get("section_title", "General Reference")
        traceability_lines.append(f"- **Page {page}** | *Section: {section}* | `{chunk_id}`")

    formatted_context = "\n\n".join(cleaned_context_list)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context Evidence:\n{formatted_context}\n\nClinical Query: {query}"},
    ]

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    global _llm_model, _llm_tokenizer, LLM_MAX_NEW_TOKENS, LLM_TEMPERATURE

    if _llm_model is None or _llm_tokenizer is None:
        raise RuntimeError("Qwen model and tokenizer not loaded. Please run cell losewnLGjaiA.")

    prompt = _llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _llm_tokenizer(prompt, return_tensors="pt").to(_llm_model.device)

    with torch.no_grad():
        output_ids = _llm_model.generate(
            **inputs,
            max_new_tokens=LLM_MAX_NEW_TOKENS,
            temperature=LLM_TEMPERATURE,
            repetition_penalty=1.15,
            pad_token_id=_llm_tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    llm_output = _llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    for pat, repl in (
        (r"###\s*Direct Clinical Answer:", "🎯 **Direct Clinical Answer**:"),
        (r"###\s*Key Guidelines & Recommendations:", "💊 **Key Guidelines & Recommendations**:"),
        (r"###\s*Considerations & Risk Factors:", "⚠️ **Considerations & Risk Factors**:"),
    ):
        llm_output = re.sub(pat, repl, llm_output, flags=re.IGNORECASE)

    traceability_block = (
        "\n\n📚 **Source Evidence & Traceability**:\n"
        + "\n".join(traceability_lines)
        + f"\n\n**Retrieval confidence:** {confidence['score']}% ({confidence['label']})"
    )
    return llm_output + traceability_block

## 4.8. Safety / Refusal Layer
This layer classifies the query's safety based on retrieval confidence, allowing us to refuse or caution against answers when evidence is insufficient.

In [82]:
def confidence_report(query, results):
    """Return a transparent retrieval-confidence estimate from multiple signals."""
    if not results:
        return {"score": 0, "label": "Low", "reason": "No evidence was retrieved."}
    top = results[0]
    dense_strength = float(np.clip((top.get("dense_score", 0.0) + 1.0) / 2.0, 0.0, 1.0))
    lexical_strength = float(top.get("lexical_score", 0.0))
    margin = float(np.clip((top.get("score", 0.0) - results[1].get("score", 0.0)) / 0.10, 0.0, 1.0)) if len(results) > 1 else 1.0
    agreement = sum(1 for r in results if r.get("lexical_score", 0.0) >= 0.20) / len(results)

    base_score = 0.60 * dense_strength + 0.28 * lexical_strength + 0.07 * margin + 0.05 * agreement

    if top.get("is_pico_annex", False):
        base_score *= 0.5

    if dense_strength > 0.80 and lexical_strength > 0.80 and not top.get("is_pico_annex", False):
        stretch_input = float(np.clip((base_score - 0.80) / 0.20, 0.0, 1.0))
        base_score = 0.90 + stretch_input * 0.08

    score = int(round(float(np.clip(100 * base_score, 0, 100))))
    if score >= MIN_CONFIDENCE_FOR_ALLOWED:
        label = "High"
    elif score >= MIN_CONFIDENCE_FOR_CAUTION:
        label = "Moderate"
    else:
        label = "Low"

    reason = (
        f"Hybrid retrieval score {top['score']:.3f}; dense/lexical agreement "
        f"{agreement:.0%}; top-result margin {margin:.2f}"
        + ("; PICO/annex penalty applied" if top.get("is_pico_annex", False) else "")
        + "."
    )
    return {"score": score, "label": label, "reason": reason}


def classify_safety(confidence):
    score = confidence["score"]
    if score < MIN_CONFIDENCE_FOR_CAUTION:
        return "Refuse", f"Retrieval confidence is low ({score}%)."
    if score < MIN_CONFIDENCE_FOR_ALLOWED:
        return "Needs Caution", f"Retrieval confidence is moderate ({score}%)."
    return "Allowed", f"Retrieval confidence is high ({score}%)."

## 4.9. End-to-End RAG Function
This section integrates retrieval, safety classification, grounded generation, and citation formatting into a single, comprehensive RAG pipeline function.

In [83]:
def format_citations(results):
    return [
        f"\U0001F4DA {DOCUMENT_NAME}\n"
        f"Section: {result['section_title']}\n"
        f"Page(s): {result['page_numbers']}\n"
        f"Chunk: {result['chunk_id']}\n"
        f"URL: {result['source_url']}"
        for result in results
    ]


def answer_question(query, top_k=DEFAULT_TOP_K):
    """End-to-end RAG: retrieve -> confidence/safety -> LLM synthesis -> format."""
    try:
        results = retrieve(query, top_k=top_k)
    except Exception as error:
        return {
            "query": query, "status": "Error",
            "answer": f"Retrieval failed: {error}", "citations": [],
        }

    confidence = confidence_report(query, results)
    status, reason = classify_safety(confidence)

    if status == "Refuse":
        return {
            "query": query, "status": status, "reason": reason,
            "confidence": confidence,
            "answer": (
                "### \u26A0\uFE0F Insufficient Evidence\n"
                "The guideline evidence is not strong enough to answer safely."
            ),
            "citations": [],
        }

    if not LLM_AVAILABLE:
        return {
            "query": query, "status": "LLM Unavailable",
            "reason": (
                "The local medical LLM did not load (no GPU runtime, or every "
                "candidate model failed to load). Enable a T4 GPU runtime "
                "(Runtime > Change runtime type > T4 GPU) and re-run Section "
                "18b before asking questions."
            ),
            "confidence": confidence,
            "answer": (
                "### \u26A0\uFE0F Answer Generation Unavailable\n"
                "**The local medical LLM is not loaded, so no synthesized answer "
                "can be produced.** Relevant source passages were retrieved -- see "
                "citations below -- but this system does not fall back to raw "
                "extracted text.\n"
            ),
            "citations": format_citations(results),
            "used_local_llm": False,
        }

    try:
        raw_answer = generate_clinical_response(
            query, retrieved_chunks=results, confidence=confidence
        )
    except Exception as error:
        return {
            "query": query, "status": "Generation Error",
            "reason": f"Local LLM generation failed: {error}",
            "confidence": confidence,
            "answer": (
                "### \u26A0\uFE0F Answer Generation Failed\n"
                f"**The local medical LLM raised an error during generation ({error}).** "
                "See citations below for the retrieved source passages.\n"
            ),
            "citations": format_citations(results),
            "used_local_llm": False,
        }

    return {
        "query": query, "status": status, "reason": reason,
        "confidence": confidence,
        "answer": raw_answer, "raw_answer": raw_answer,
        "used_local_llm": True, "citations": [],
        "retrieved_chunks": [
            {
                "chunk_id": result["chunk_id"],
                "section_title": result["section_title"],
                "page_numbers": result["page_numbers"],
                "score": result["score"],
            }
            for result in results
        ],
    }

# 5. Demonstration and Evaluation

This section showcases the RAG system's capabilities through demo questions and provides a summary of its performance.

## 5.1. Demo Questions
Here, we test the end-to-end RAG system with a set of sample questions to demonstrate its functionality.

In [84]:
DEMO_QUESTIONS = [
    "What is the recommended first-line pharmacological treatment for hypertension?",
    "When should blood pressure be reassessed after initiating pharmacological treatment?",
    "What are the target blood pressure levels for adults?",
]

for question in DEMO_QUESTIONS:
    result = answer_question(question)
    print("\n" + "\u2550" * 88)
    print(f"\u2753 QUESTION\n{result['query']}")
    # Removed hardcoded emojis for status, confidence, and engine as the LLM will generate them.
    # The full formatted answer (including emojis) is in result['answer'].
    print(f"\n{result['answer']}")


════════════════════════════════════════════════════════════════════════════════════════
❓ QUESTION
What is the recommended first-line pharmacological treatment for hypertension?

The WHO recommendation for adults with hypertension requiring pharmacological treatment is to use any of the following three classes of pharmacological antihypertensive medications as an initial treatment: thiazide and thiazide-like agents, angiotensin-converting enzyme inhibitors (ACEi)/angiotensin receptor blockers (ARBs), and long-acting dihydropyridine calcium channel blockers (CCBs). These drugs are recommended over placebo due to the absence of data on cardiovascular event effects. Additionally, the choice between a single pill combination versus a combination therapy of multiple drugs depends on various factors such as patient characteristics, existing conditions, and baseline blood pressure levels.

📚 **Source Evidence & Traceability**:
- **Page 10, 23** | *Section: 4. RECOMMENDATION ON DRUG CLASSES 

### Run Demo Questions through RAG Pipeline

This cell iterates through a predefined list of `DEMO_QUESTIONS` and passes each one to the `answer_question` function, which orchestrates the entire RAG pipeline. It then prints the question and the comprehensive answer generated by the system, including traceability information and confidence scores, showcasing the end-to-end functionality.

## 5.2. Evaluation Summary
This section provides a consolidated overview of the current evaluation results and key pipeline metrics.

### Pipeline Summary and Heuristic Evaluation

This cell provides a comprehensive overview of the data processing and RAG pipeline statistics. It includes metrics like the number of cleaned pages, total blocks, chunks, and details about the embedding model. Additionally, it presents a heuristic evaluation of the retrieval performance, highlighting mean precision and keyword coverage.

In [85]:
print("🔎 SAMPLE HYBRID RETRIEVAL")
for result in sample_results[:5]:
    print(
        f"{result['chunk_id']}: combined={result['score']:.3f}, "
        f"dense={result['dense_score']:.3f}, lexical={result['lexical_score']:.3f}"
    )

🔎 SAMPLE HYBRID RETRIEVAL
chunk_107: combined=0.852, dense=0.546, lexical=1.000
chunk_111: combined=0.808, dense=0.596, lexical=0.825
chunk_109: combined=0.796, dense=0.559, lexical=0.825


### Display Retrieval Evaluation Summary

This cell prints a concise summary of the pipeline's key statistics and the results of the heuristic retrieval evaluation. It includes metrics like the number of processed pages, blocks, and chunks, details about the embedding model, and the mean precision and keyword coverage from the heuristic tests. This provides a quick overview of the system's performance.

In [86]:
print("🧪 PIPELINE SUMMARY")
print("-" * 44)
print("Cleaned pages        :", len(safe_cleaned_pages))
print("Total blocks         :", len(all_blocks))
print("Total chunks         :", len(final_chunks))
print("Embedding model      :", embedding_model_name)
print("Embedding dimension  :", embedding_dim)
print("FAISS vectors        :", faiss_index.ntotal)
print()
print("📈 HEURISTIC RETRIEVAL EVALUATION")
print("These scores are retrieval diagnostics, not human-labeled ground truth.")
print(f"Mean Precision@{DEFAULT_TOP_K}: {mean_precision:.2f}")
print(f"Mean keyword coverage : {mean_coverage:.2f}")
for result in eval_results:
    print(
        f"  - {result['query'][:60]:<60} "
        f"P@{DEFAULT_TOP_K}={result['heuristic_precision_at_k']:.2f} "
        f"coverage={result['mean_keyword_coverage']:.2f}"
    )

🧪 PIPELINE SUMMARY
--------------------------------------------
Cleaned pages        : 52
Total blocks         : 494
Total chunks         : 111
Embedding model      : sentence-transformers/embeddinggemma-300m-medical
Embedding dimension  : 768
FAISS vectors        : 111

📈 HEURISTIC RETRIEVAL EVALUATION
These scores are retrieval diagnostics, not human-labeled ground truth.
Mean Precision@5: 1.00
Mean keyword coverage : 0.81
  - What is the recommended first-line pharmacological treatment P@5=1.00 coverage=0.90
  - When should blood pressure be reassessed after initiating ph P@5=1.00 coverage=0.80
  - What are the adverse effects of antihypertensive medications P@5=1.00 coverage=0.67
  - What is the target blood pressure for adults with hypertensi P@5=1.00 coverage=0.87


# 6. Conclusion & Next Steps

This final section summarizes the project's findings, discusses its limitations, and proposes future directions for improvement and expansion.

### Conclusion

This notebook successfully demonstrates an end-to-end Retrieval Augmented Generation (RAG) pipeline tailored for medical guidelines, specifically using the WHO hypertension guideline. We've implemented robust OCR-aware text cleaning, section-aware chunking, and a hybrid retrieval mechanism that combines semantic and lexical relevance. The system culminates in grounded, evidence-based answer generation using a local medical LLM.

Key achievements include:

*   **Data Integrity**: Ensuring high-quality input through meticulous cleaning and validation of OCR data.
*   **Contextual Retrieval**: Employing section-aware chunking and hybrid retrieval to fetch relevant and precise evidence.
*   **Local LLM Integration**: Utilizing a specialized medical LLM for nuanced and medically accurate answer synthesis without external API dependencies.
*   **Traceability**: Providing clear citations and confidence scores for generated answers, enhancing trustworthiness.

### Next Steps

Potential future enhancements include:

*   **Advanced Semantic Chunking**: Exploring more sophisticated methods to preserve document hierarchy and discourse structure.
*   **Dynamic LLM Selection**: Implementing a mechanism to automatically select the optimal local LLM based on task requirements and available resources.
*   **User Feedback Loop**: Incorporating user feedback to continuously improve retrieval relevance and answer quality.
*   **Broader Application**: Extending the RAG pipeline to other medical guidelines and clinical documents to further validate its versatility and effectiveness.